# Tutorial B02: Zarr Time Series Inspection and Band Diagnostics

This notebook is a diagnostic tool for inspecting S1-GRiTS Zarr time series stores.
It helps identify NaN patterns, data gaps, and GLCM texture band discontinuities.

**What this notebook covers:**

1. Open a Zarr store and inspect all band names, shapes, and dtypes
2. Compute NaN statistics per band across the full spatial extent
3. Plot per-timestep NaN fraction for GLCM bands
4. Sample a center pixel and plot all bands as time series
5. Compare spatial NaN maps between VV_dB and GLCM bands
6. Overlay GLCM NaN fraction against monthly scene count to diagnose root cause

**Typical use case:** After processing, run this notebook to verify data completeness
and diagnose any unexpected NaN spikes or temporal discontinuities.


In [ ]:
# =============================================================================
# Configuration
# =============================================================================

# Path to the Zarr store for the tile you want to inspect
ZARR_PATH = r"YOUR_ZARR_PATH"   # e.g. r"D:/output/17MPU_ASCENDING/zarr/S1_monthly.zarr"

# Pixel to sample for time series (row, col) -- None = auto center
SAMPLE_PIXEL = None

# Additional pixels to compare (list of (row, col) tuples)
# Useful to check if NaN pattern is pixel-specific or global
EXTRA_PIXELS = []  # e.g. [(100, 100), (200, 300)]


## 1. Open Zarr and Inspect Structure


In [ ]:
# =============================================================================
# Open Zarr and inspect structure
# =============================================================================
import zarr
import numpy as np
import pandas as pd

z = zarr.open(ZARR_PATH, mode="r")

print("=== Zarr Store Structure ===")
band_keys = []
for k in sorted(z.keys()):
    arr = z[k]
    print(f"  {k:40s} shape={arr.shape}  dtype={arr.dtype}")
    if arr.ndim == 3:
        band_keys.append(k)

# Parse time axis
if "time" in z:
    raw_times = z["time"][:]
    try:
        times = pd.to_datetime(raw_times)
    except Exception:
        times = pd.RangeIndex(len(raw_times))
    print(f"
Time steps: {len(times)}  ({times[0]} ... {times[-1]})")
else:
    n_time = z[band_keys[0]].shape[0] if band_keys else 0
    times = pd.RangeIndex(n_time)
    print(f"
No time key found. Using index 0..{n_time-1}")

print(f"
3-D band keys ({len(band_keys)}): {band_keys}")


## 2. NaN Statistics Per Band


In [ ]:
# =============================================================================
# NaN statistics per band (full spatial extent)
# =============================================================================
print(f"{chr(34)}Band{chr(34):40s} {chr(34)}NaN%{chr(34):>7s}  {chr(34)}min{chr(34):>10s}  {chr(34)}max{chr(34):>10s}  {chr(34)}mean{chr(34):>10s}")
print("-" * 85)

nan_stats = {}
for k in sorted(band_keys):
    arr = z[k][:].astype(np.float32)          # load full array (time, y, x)
    arr = np.where(arr < -9998.0, np.nan, arr) # -9999 -> NaN
    total = arr.size
    n_nan = int(np.sum(np.isnan(arr)))
    pct   = 100.0 * n_nan / total
    vmin  = float(np.nanmin(arr)) if n_nan < total else float("nan")
    vmax  = float(np.nanmax(arr)) if n_nan < total else float("nan")
    vmean = float(np.nanmean(arr)) if n_nan < total else float("nan")
    nan_stats[k] = pct
    print(f"  {k:38s} {pct:6.1f}%  {vmin:10.4f}  {vmax:10.4f}  {vmean:10.4f}")

print()
# Separate core bands vs GLCM bands
core_bands = [k for k in band_keys if "glcm" not in k.lower()]
glcm_bands = [k for k in band_keys if "glcm" in k.lower()]
print(f"Core bands: {core_bands}")
print(f"GLCM bands: {glcm_bands}")
